[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_04/1_algorithms.ipynb)

# Day 4: Algorithms (easy)

**How to work:** Name the pattern before you code. Stuck 15 min: open Hint 1. Stuck 25 min: Hint 2, then the solution. Say the time and space complexity out loud. Re-type the solution from memory if you needed it.

**Today (52 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Totals so far | easy | 10 |
| Q2 | Zip two sorted chains | easy | 15 |
| Q3 | Push the zeros to the back | easy | 12 |
| Q4 | Widest path in a tree (bonus) | easy | 15 |

**Handbook:** [Algorithms](https://mahsa-agz.github.io/ds-handbook/#algo), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import copy

def check(fn, cases, key=None):
    """Runs fn on each (args, expected) case and prints PASS or FAIL. key: normalise outputs (e.g. sorted)."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            got = "error: " + repr(e)
        good = (key(got) == key(want)) if (key and not isinstance(got, str)) else got == want
        ok += good
        shown = ", ".join(repr(a) for a in args)
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}({shown}) -> {got!r}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

def build_list(vals):
    """[1, 2, 3] -> 1 -> 2 -> 3 (returns the head, or None for an empty list)."""
    dummy = tail = ListNode()
    for v in vals:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def list_values(head, limit=10000):
    """Linked list -> python list (stops after `limit` nodes in case of a cycle)."""
    out = []
    while head is not None and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(vals):
    """Level-order list with None for gaps (LeetCode style) -> root. [1, 2, 3, None, 4] means 2 has a right child 4."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_values(root):
    """Root -> level-order list with None for gaps (trailing None removed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if not isinstance(node, TreeNode):
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

The setup cell gives you `ListNode`, `build_list`, `list_values`, `TreeNode`, `build_tree` (level order,
`None` = no child) and `tree_values`. The tests use them to convert plain lists to nodes and back.

---
### Q1. Totals so far

*easy, about 10 min*

Given `nums`, return a list where position `i` holds `nums[0] + nums[1] + ... + nums[i]`. Then answer
out loud: once you have this list, how do you get the sum of `nums[i..j]` in O(1)?

Examples:
- `[3, 1, 2, 10, 1]` -> `[3, 4, 6, 16, 17]`
- `[1, 1, 1]` -> `[1, 2, 3]`
- `[]` -> `[]`

*Source: [LeetCode 1480](https://leetcode.com/problems/running-sum-of-1d-array/)*

In [ ]:
def running_sum(nums):
    pass

# tests
check(running_sum, [
    ([1, 2, 3, 4], [1, 3, 6, 10]),
    ([1, 1, 1, 1, 1], [1, 2, 3, 4, 5]),
    ([3, 1, 2, 10, 1], [3, 4, 6, 16, 17]),
    ([], []),
    ([5], [5]),
    ([-2, 5, -1], [-2, 3, 2]),
])

<details><summary><b>Hint 1</b></summary>

Signal: each answer is the previous answer plus one new number. Pattern: **prefix sum**.

</details>

<details><summary><b>Hint 2</b></summary>

1. `total = 0`, `out = []`.
2. For each `x`: `total += x`, append `total`.
3. Follow-up: with `pre[0] = 0` and `pre[i+1] = nums[0] + ... + nums[i]`, the sum of `nums[i..j]` is
   `pre[j+1] - pre[i]`.

</details>

<details><summary><b>Solution</b></summary>

```python
def running_sum(nums):
    out, total = [], 0
    for x in nums:
        total += x
        out.append(total)
    return out

# same thing with the standard library
from itertools import accumulate
print(list(accumulate([3, 1, 2, 10, 1])))   # [3, 4, 6, 16, 17]
```

**Why:** Recomputing each total from scratch is O(n^2). Carrying the total forward makes it O(n). This list is the building block for O(1) range sums (`pre[j+1] - pre[i]`), which comes back on later days. In pandas the same idea is `df['x'].cumsum()`, and in SQL it is `SUM(x) OVER (ORDER BY t)`.

**Complexity:** O(n) time, O(n) space for the output (O(1) extra if you overwrite `nums` in place).

**Common mistakes:** Off-by-one in the range-sum formula (forgetting the leading 0). Writing a nested loop.

**Learn more:** [algo-prefix-sum](https://mahsa-agz.github.io/ds-handbook/#algo-prefix-sum)

</details>

---
### Q2. Zip two sorted chains

*easy, about 15 min*

You get the heads of two linked lists, each sorted in increasing order. Splice their nodes together into
one sorted list (reuse the nodes, do not create new values) and return its head.

Examples (values):
- `1 -> 2 -> 4` and `1 -> 3 -> 4` -> `1 -> 1 -> 2 -> 3 -> 4 -> 4`
- empty and `0` -> `0`
- `5` and `1 -> 2 -> 3` -> `1 -> 2 -> 3 -> 5`

*Source: [LeetCode 21](https://leetcode.com/problems/merge-two-sorted-lists/)*

In [ ]:
def merge_two_lists(l1, l2):
    pass

# tests
def merged_values(a, b):
    return list_values(merge_two_lists(build_list(a), build_list(b)))

check(merged_values, [
    (([1, 2, 4], [1, 3, 4]), [1, 1, 2, 3, 4, 4]),
    (([], []), []),
    (([], [0]), [0]),
    (([5], [1, 2, 3]), [1, 2, 3, 5]),
    (([1, 1], [1]), [1, 1, 1]),
    (([-3, 0, 7], [-5, 8]), [-5, -3, 0, 7, 8]),
])

<details><summary><b>Hint 1</b></summary>

Signal: two sorted sequences, always take the smaller front item. Pattern: **linked list merge with a dummy head** (the merge step of merge sort).

</details>

<details><summary><b>Hint 2</b></summary>

1. Create `dummy = ListNode()` and `tail = dummy`.
2. While both lists are non-empty: attach the smaller head to `tail.next`, advance that list, advance `tail`.
3. Attach whatever is left (`l1 or l2`) to `tail.next`.
4. Return `dummy.next`.

</details>

<details><summary><b>Solution</b></summary>

```python
def merge_two_lists(l1, l2):
    dummy = tail = ListNode()
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next, l1 = l1, l1.next
        else:
            tail.next, l2 = l2, l2.next
        tail = tail.next
    tail.next = l1 or l2          # one list is empty; the other is already sorted
    return dummy.next
```

**Why:** The smallest remaining value is always at the front of one of the two lists, so one comparison decides each node. The dummy head removes the special case of choosing the first node. The leftover tail can be attached in one step because it is already sorted.

**Complexity:** O(n + m) time, O(1) extra space (nodes are reused).

**Common mistakes:** Forgetting to attach the leftover list. Returning `dummy` instead of `dummy.next`. Creating new nodes (works, but uses O(n + m) space and is not what was asked).

**Learn more:** [algo-linked-list](https://mahsa-agz.github.io/ds-handbook/#algo-linked-list)

</details>

---
### Q3. Push the zeros to the back

*easy, about 12 min*

Move all `0` values of `nums` to the end **in place**, keeping the other values in their original order.
Return nothing; the test reads `nums` afterwards. Do not make a copy of the list.

Examples:
- `[0, 1, 0, 3, 12]` -> `[1, 3, 12, 0, 0]`
- `[4, 0, 5]` -> `[4, 5, 0]`
- `[0]` -> `[0]`

*Source: [LeetCode 283](https://leetcode.com/problems/move-zeroes/)*

In [ ]:
def move_zeroes(nums):
    pass

# tests
def after_move(nums):
    move_zeroes(nums)
    return nums

check(after_move, [
    ([0, 1, 0, 3, 12], [1, 3, 12, 0, 0]),
    ([0], [0]),
    ([4, 0, 5], [4, 5, 0]),
    ([1, 2, 3], [1, 2, 3]),
    ([0, 0, 1], [1, 0, 0]),
    ([2, 0, 0, -1, 0, 7], [2, -1, 7, 0, 0, 0]),
])

<details><summary><b>Hint 1</b></summary>

Signal: rearrange in place while keeping relative order. Pattern: **two pointers** (a slow write pointer and a fast read pointer).

</details>

<details><summary><b>Hint 2</b></summary>

1. `write = 0`.
2. For each `read` index: if `nums[read] != 0`, swap `nums[write]` and `nums[read]`, then `write += 1`.
3. Everything from `write` onward is now 0.

</details>

<details><summary><b>Solution</b></summary>

```python
def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1
```

**Why:** `write` marks where the next non-zero value belongs. The read pointer finds non-zeros in their original order, so their order is kept, and the swap pushes zeros behind them. Every element is read once.

**Complexity:** O(n) time, O(1) space.

**Common mistakes:** Removing zeros with `nums.remove(0)` inside a loop (O(n^2) and skips elements). Building a new list and assigning `nums = ...` (only rebinds the local name; use `nums[:] = ...` if you must).

**Learn more:** [algo-two-pointers](https://mahsa-agz.github.io/ds-handbook/#algo-two-pointers)

</details>

---
### Q4. Widest path in a tree (bonus)

*easy, about 15 min*

Optional. Return the length, counted in edges, of the longest path between any two nodes of a binary
tree. The path does not have to pass through the root.

Examples (level order, `None` = missing child):
- `[1, 2, 3, 4, 5]` -> `3` (path 4, 2, 1, 3)
- `[1, 2]` -> `1`
- `[1, 2, None, 3, 4, 5, None, None, 6]` -> `4` (path 5, 3, 2, 4, 6, which skips the root)

*Source: [LeetCode 543](https://leetcode.com/problems/diameter-of-binary-tree/)*

In [ ]:
def diameter(root):
    pass

# tests
def diameter_of(vals):
    return diameter(build_tree(vals))

check(diameter_of, [
    ([1, 2, 3, 4, 5], 3),
    ([1, 2], 1),
    ([1], 0),
    ([], 0),
    ([1, 2, None, 3, 4, 5, None, None, 6], 4),
    ([1, 2, 3, 4, None, None, 5, 6, None, None, 7], 6),
])

<details><summary><b>Hint 1</b></summary>

Signal: a value for every node (its height) plus a best answer that can be anywhere. Pattern: **tree DFS that returns height and updates a global best**.

</details>

<details><summary><b>Hint 2</b></summary>

1. Write `height(node)`: 0 for None, else `1 + max(height(left), height(right))`.
2. Inside it, before returning, update `best = max(best, height(left) + height(right))`: the longest path that
   bends at this node.
3. Call it on the root and return `best`.

</details>

<details><summary><b>Solution</b></summary>

```python
def diameter(root):
    best = 0
    def height(node):
        nonlocal best
        if node is None:
            return 0
        left, right = height(node.left), height(node.right)
        best = max(best, left + right)       # path that bends at this node
        return 1 + max(left, right)
    height(root)
    return best
```

**Why:** Every path has one highest node where it bends; there it uses the left height plus the right height. Computing heights bottom-up visits each node once and checks every possible bend point. Calling a separate height function from every node would be O(n^2).

**Complexity:** O(n) time, O(h) recursion space.

**Common mistakes:** Returning `left + right` at the root only (fails when the best path skips the root). Counting nodes instead of edges. Forgetting `nonlocal`.

**Learn more:** [algo-trees](https://mahsa-agz.github.io/ds-handbook/#algo-trees)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [SQL](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_04/2_sql.ipynb)